# 1.5 Lab - Building a Multi-Source E-commerce Pipeline with SDP

You are the data engineer for an online electronics shop. Orders arrive from two channels in **different formats with different column names**, customers change their details in a CRM, and the business wants trustworthy revenue reporting.

**You will build** (in `01-advanced-sdp/pipelines/lab_ecommerce/transformations/`):

| Step | Dataset | Technique |
|---|---|---|
| 1 | `orders_bronze` | Multi-flow: EU CSV + US JSON → one unified streaming table, clustered by `source_region` |
| 2 | `products`, `customers_cdc`, `customers` | Materialized view; AUTO CDC → **SCD type 2** dimension |
| 3 | `orders_silver`, `orders_quarantine` | `expect_all_or_drop` + quarantine of rejected rows, Liquid Clustering |
| 3 | `revenue_by_category_daily`, `revenue_by_customer_tier` | Gold materialized views |

![Lab target graph: orders_eu CSV and orders_us JSON feed orders_eu_flow and orders_us_flow into orders_bronze; products CSV becomes the products materialized view; customers_cdc JSON becomes the customers_cdc streaming table and an AUTO CDC flow builds the SCD2 customers table; orders_silver and orders_quarantine read orders_bronze; revenue_by_category_daily joins products and revenue_by_customer_tier joins current customers](../images/1.5-lab-target-graph.svg)

**How to read it:** this is the pipeline graph you should see once all TODOs are done. The STEP tags match the Run/Check cells below: each step adds one source file and the boxes it defines. The check in step 3 tests the arrows out of `orders_bronze`: every Bronze row must land in `orders_silver` or `orders_quarantine`.

**How to work:** open the three files in your editor, complete the `TODO`s, then run the matching **Run** and **Check** cells below. Each step only loads the files completed so far.

Stuck? The reference solution is in `01-advanced-sdp/solutions/lab_ecommerce/`. Set `USE_SOLUTION = True` to run it.

In [ ]:
USE_SOLUTION = False

In [ ]:
from adelab.classroom import setup
from adelab import datagen
from pyspark.sql import functions as F

DA = setup("1.5", reset=True)
datagen.land_ecommerce(DA.paths.landing, batch=0)

ROOT = "solutions/lab_ecommerce" if USE_SOLUTION else "pipelines/lab_ecommerce"
CONF = {
    "orders_eu_source": f"{DA.paths.landing}/orders_eu",
    "orders_us_source": f"{DA.paths.landing}/orders_us",
    "products_source": f"{DA.paths.landing}/products",
    "customers_source": f"{DA.paths.landing}/customers_cdc",
}

def lab_pipeline(steps: int):
    files = ["01_bronze_orders.py", "02_dimensions.py", "03_silver_gold.py"][:steps]
    return DA.pipeline("lab_ecommerce", source=[f"{ROOT}/transformations/{f}" for f in files],
                       configuration=CONF)

## Explore the sources first
Look at the column names and types. They differ between channels, and the EU CSV is all strings with some dirty values.

In [ ]:
display(spark.read.option("header", True).csv(CONF["orders_eu_source"]).limit(5))
display(spark.read.json(CONF["orders_us_source"]).limit(5))
display(spark.read.json(CONF["customers_source"]).limit(5))

## Step 1 - Unified Bronze (multi-flow)
Complete **TODO 1.1 - 1.3** in `01_bronze_orders.py`, then run:

In [ ]:
update = lab_pipeline(1).run()

In [ ]:
# CHECK step 1
b = spark.table("orders_bronze")
expected = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "order_ts", "source_region", "source_file"]
assert b.columns == expected, f"columns should be {expected}, got {b.columns}"
assert dict(b.groupBy("source_region").count().collect()) == {"EU": 120, "US": 100}, "expected 120 EU + 100 US rows"
types = dict(b.dtypes)
assert types["customer_id"] == "bigint" and types["quantity"] == "int" and types["unit_price"] == "double"     and types["order_ts"] == "timestamp", f"cast the columns to the target schema, got {types}"
detail = spark.sql("DESCRIBE DETAIL orders_bronze").first()
assert list(detail["clusteringColumns"]) == ["source_region"], "orders_bronze must be clustered by source_region"
print("Step 1 passed")

## Step 2 - Dimensions
Complete **TODO 2.1 - 2.3** in `02_dimensions.py`, then run:

In [ ]:
update = lab_pipeline(2).run()

In [ ]:
# CHECK step 2
assert spark.table("products").count() == 30, f"products: expected 30 rows, got {spark.table('products').count()}"
c = spark.table("customers")
assert {"__START_AT", "__END_AT"} <= set(c.columns), "customers must be SCD type 2 (stored_as_scd_type=2)"
assert not {"operation", "sequence_num"} & set(c.columns), "exclude operation and sequence_num (except_column_list)"
assert c.count() == 40 and c.filter("__END_AT IS NULL").count() == 40,     f"expected 40 customers, all with an open version; got {c.count()} rows"
print("Step 2 passed")

## Step 3 - Silver, quarantine, and Gold
Complete **TODO 3.1 - 3.5** in `03_silver_gold.py`, then run:

In [ ]:
update = lab_pipeline(3).run()

In [ ]:
# CHECK step 3
bronze, silver, quarantine = (spark.table(t).count() for t in ("orders_bronze", "orders_silver", "orders_quarantine"))
assert bronze == silver + quarantine, f"no row may be lost: bronze={bronze} silver={silver} quarantine={quarantine}"
assert quarantine > 0 and "failed_rules" in spark.table("orders_quarantine").columns,     "orders_quarantine needs the rejected rows and a failed_rules column"
bad = spark.table("orders_silver").filter(
    "NOT (quantity > 0 AND unit_price > 0 AND customer_id IS NOT NULL AND order_ts IS NOT NULL) "
    "OR quantity IS NULL OR unit_price IS NULL").count()
assert bad == 0, f"{bad} invalid rows reached orders_silver: check the rules of TODO 3.1 (NULLs included)"
assert list(spark.sql("DESCRIBE DETAIL orders_silver").first()["clusteringColumns"]) == ["order_date", "source_region"],     "orders_silver must be clustered by (order_date, source_region)"
gold = spark.table("revenue_by_category_daily")
assert set(gold.columns) >= {"order_date", "category", "orders", "revenue"}, f"columns: {gold.columns}"
tiers = {r["tier"] for r in spark.table("revenue_by_customer_tier").collect()}
assert tiers <= {"bronze", "silver", "gold", "unknown"} and len(tiers) >= 3, f"tiers: {tiers}"
print("Step 3 passed")

## Step 4 - Day 2: incremental data and history
A second day of data arrives: new orders and customer changes (some customers move or change tier, some are deleted). Run the **same** pipeline. Only new files are processed.

In [ ]:
datagen.land_ecommerce(DA.paths.landing, batch=1)
update = lab_pipeline(3).run()

In [ ]:
# CHECK step 4
counts = dict(spark.table("orders_bronze").groupBy("source_region").count().collect())
assert counts == {"EU": 240, "US": 200}, f"each file must be read exactly once: {counts}"
history = spark.sql("SELECT customer_id FROM customers GROUP BY customer_id HAVING count(*) > 1").count()
assert history > 0, "some customers should now have more than one version"
print(f"Step 4 passed - {history} customers have history")

In [ ]:
%%sql
SELECT customer_id, name, city, tier, __START_AT, __END_AT
FROM customers
WHERE customer_id IN (SELECT customer_id FROM customers GROUP BY customer_id HAVING count(*) > 1)
ORDER BY customer_id, __START_AT

In [ ]:
display(spark.table("revenue_by_customer_tier").orderBy(F.desc("revenue")))
display(lab_pipeline(3).expectation_metrics(latest_only=False))

## Reflection questions
1. Why must each source get its **own append flow** instead of one `unionByName` query? (Think about checkpoints, adding a third channel, and backfilling one channel.)
2. `revenue_by_customer_tier` joins orders to the **current** customer version. How would you attribute revenue to the tier a customer had **at order time** instead? (Hint: join on `order_ts` between `__START_AT` and `__END_AT`, using a timestamp `sequence_by`.)
3. The EU feed sends `unit_price = "N/A"`. Where is that row now, and how would you reprocess it after the source fixes the data?

In [ ]:
DA.cleanup()